# S1 → M1 Connectivity: Rasters & PSTHs from Human ICMS Data
## MaDeLaNe 2026 — Multi-Area High-Density Laminar Neurophysiology

This notebook visualises raw spike data from **Shelchkova et al. (2023)** *Nature Communications*:
*"Microstimulation of human somatosensory cortex evokes task-dependent,
spatially patterned responses in motor cortex"*

**Experiment overview**

| Item | Detail |
|------|--------|
| Participants | 3 humans with spinal-cord injury (BCI clinical trial) |
| Stimulation | ICMS through individual electrodes in S1 (area 1) |
| Recording | Single-unit / multi-unit activity on Utah arrays in M1 |
| Stim parameters | 60 uA, 100 Hz, 1-s trains (20 repetitions per S1 electrode) |
| Raw data format | One `.mat` (HDF5 / v7.3) file per S1-electrode x M1-channel pair |

**Data source**: [DABI repository](https://dabi.loni.usc.edu/) — Connectivity Paper, S1M1 Data, RawData.

Each file is named `SE<n>_MC<m>_100Hz_Raw_<date>.mat` where SE = S1 stimulation electrode, MC = M1 recording channel.


In [ ]:
# @title 1. Install dependencies & set data path
!pip install h5py numpy matplotlib scipy -q

import os

# -- Point this at your copy of the RawData/C1 folder --
# Option A: Colab + Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/MaDeLaNe/Shelchkova_etal/DABI/ConnectivityPaper/S1M1 Data/RawData/C1"

# Option B: uploaded / local
DATA_DIR = "/content/RawData/C1"   # <-- adjust as needed

assert os.path.isdir(DATA_DIR), f"Data directory not found: {DATA_DIR}"
print(f"Data directory: {DATA_DIR}")
print(f"  Files: {len(os.listdir(DATA_DIR)):,}")


In [ ]:
# @title 2. Data loader
import glob
import h5py
import numpy as np

def load_channel_pair(data_dir, stim_elec, motor_ch, freq=100):
    """Load raw spike + stimulation data for one SE x MC pair.

    Returns a dict with spike_times, train_starts, train_ends,
    train_amp, train_freq, n_trains, stim_elec, motor_ch.
    """
    pattern = f"{data_dir}/SE{stim_elec}_MC{motor_ch}_{freq}Hz_Raw_*.mat"
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No file matching {pattern}")

    with h5py.File(files[0], 'r') as f:
        sm = f['smRaw']

        spike_times = sm['spike']['time'][:].flatten()
        n_trains    = int(sm['stim']['train']['numTrains'][0, 0])
        train_amp   = sm['stim']['train']['amp'][:].flatten()
        train_freq  = sm['stim']['train']['freq'][:].flatten()

        # Per-train pulse times are stored as HDF5 object references
        refs = sm['stim']['train']['time'][0]
        train_starts, train_ends = [], []
        for ref in refs:
            t = f[ref][:].flatten()
            train_starts.append(t[0])
            train_ends.append(t[-1])

    return dict(
        spike_times  = spike_times,
        n_trains     = n_trains,
        train_starts = np.array(train_starts),
        train_ends   = np.array(train_ends),
        train_amp    = train_amp,
        train_freq   = train_freq,
        stim_elec    = stim_elec,
        motor_ch     = motor_ch,
    )

print("load_channel_pair() defined")


In [ ]:
# @title 3. Plotting helpers
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d


def plot_raster_and_psth(data, window=(-0.5, 1.5), bin_size=0.02,
                         sigma_ms=30, title=None,
                         ax_raster=None, ax_psth=None):
    """Train-aligned raster (top) + PSTH with Gaussian-smoothed SDF (bottom)."""
    spike_times  = data['spike_times']
    train_starts = data['train_starts']
    n_trains     = data['n_trains']

    if ax_raster is None or ax_psth is None:
        fig, (ax_raster, ax_psth) = plt.subplots(
            2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])

    # -- Raster --
    all_aligned = []
    for i, t0 in enumerate(train_starts):
        mask    = (spike_times >= t0 + window[0]) & (spike_times < t0 + window[1])
        aligned = spike_times[mask] - t0
        all_aligned.append(aligned)
        ax_raster.scatter(aligned, np.full_like(aligned, i),
                          marker='|', s=18, c='k', linewidths=0.6)

    ax_raster.axvline(0, color='red', ls='--', alpha=0.7, label='Stim onset')
    ax_raster.axvspan(0, 1.0, color='red', alpha=0.08, label='Stim (1 s)')
    ax_raster.set_ylabel('Trial (train #)')
    ax_raster.set_ylim(-0.5, n_trains - 0.5)
    ax_raster.legend(loc='upper right', fontsize=8)
    if title:
        ax_raster.set_title(title, fontsize=12, fontweight='bold')

    # -- PSTH / SDF --
    all_spikes  = np.concatenate(all_aligned)
    bins        = np.arange(window[0], window[1] + bin_size, bin_size)
    counts, be  = np.histogram(all_spikes, bins=bins)
    rate        = counts / (bin_size * n_trains)
    bc          = (be[:-1] + be[1:]) / 2
    sigma_bins  = (sigma_ms / 1000) / bin_size
    sdf         = gaussian_filter1d(rate.astype(float), sigma=sigma_bins)

    ax_psth.bar(bc, rate, width=bin_size, alpha=0.3, color='gray', label='PSTH')
    ax_psth.plot(bc, sdf, 'b-', lw=1.5, label=f'SDF (sigma = {sigma_ms} ms)')
    ax_psth.axvline(0, color='red', ls='--', alpha=0.7)
    ax_psth.axvspan(0, 1.0, color='red', alpha=0.08)

    # baseline reference
    bl_mask = (bc >= window[0]) & (bc < 0)
    if bl_mask.sum() > 0:
        bl = rate[bl_mask].mean()
        ax_psth.axhline(bl, color='green', ls=':', alpha=0.6,
                        label=f'Baseline = {bl:.1f} Hz')

    ax_psth.set_xlabel('Time relative to stim onset (s)')
    ax_psth.set_ylabel('Firing rate (Hz)')
    ax_psth.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    return ax_raster, ax_psth


def plot_pulse_locked(data, window_ms=(-2, 9), bin_size_ms=0.25,
                      max_pulses=600, title=None):
    """Raster + PSTH aligned to individual ICMS pulses (ms timescale).

    The paper discards the first 2 ms post-pulse to avoid stimulation
    artefact.  Short-latency, pulse-locked responses at 2-7 ms suggest
    monosynaptic S1 -> M1 connections.
    """
    spike_times = data['spike_times']

    # Reconstruct all pulse times (100 Hz = 10 ms IPI)
    pulse_times = np.concatenate(
        [t0 + np.arange(100) * 0.01 for t0 in data['train_starts']])

    # Subsample for plotting
    if len(pulse_times) > max_pulses:
        idx = np.linspace(0, len(pulse_times) - 1, max_pulses).astype(int)
        pt_plot = pulse_times[idx]
    else:
        pt_plot = pulse_times

    fig, (ax_r, ax_p) = plt.subplots(
        2, 1, figsize=(10, 6), sharex=True, height_ratios=[2, 1])

    ws = (window_ms[0] / 1000, window_ms[1] / 1000)
    all_aligned = []
    for i, t0 in enumerate(pt_plot):
        mask = (spike_times >= t0 + ws[0]) & (spike_times < t0 + ws[1])
        a_ms = (spike_times[mask] - t0) * 1000
        all_aligned.append(a_ms)
        ax_r.scatter(a_ms, np.full_like(a_ms, i),
                     marker='|', s=8, c='k', linewidths=0.3)

    ax_r.axvline(0, color='red', alpha=0.7)
    ax_r.axvspan(0, 2, color='gray', alpha=0.2, label='Artefact zone (0-2 ms)')
    ax_r.set_ylabel('Pulse #')
    ax_r.legend(loc='upper right', fontsize=8)
    if title:
        ax_r.set_title(title, fontsize=12, fontweight='bold')

    # PSTH
    all_ms = np.concatenate(all_aligned) if all_aligned else np.array([])
    bins   = np.arange(window_ms[0], window_ms[1] + bin_size_ms, bin_size_ms)
    if len(all_ms) > 0:
        counts, be = np.histogram(all_ms, bins=bins)
        rate = counts / ((bin_size_ms / 1000) * len(pt_plot))
        bc   = (be[:-1] + be[1:]) / 2
        ax_p.bar(bc, rate, width=bin_size_ms, color='steelblue', alpha=0.7)

    ax_p.axvline(0, color='red', alpha=0.7)
    ax_p.axvspan(0, 2, color='gray', alpha=0.2)
    ax_p.set_xlabel('Time relative to pulse (ms)')
    ax_p.set_ylabel('Firing rate (Hz)')
    plt.tight_layout()
    return fig


print("plot_raster_and_psth() defined")
print("plot_pulse_locked() defined")


---
## Example 1: High-activity channel — SE1 x MC56

MC56 has ~4,900 spikes across the session, making it one of the most active
M1 units during SE1 stimulation.  With 20 trials and a high baseline rate,
this channel shows a well-resolved SDF and dense raster.


In [ ]:
data_mc56 = load_channel_pair(DATA_DIR, stim_elec=1, motor_ch=56)
print(f"SE{data_mc56['stim_elec']} x MC{data_mc56['motor_ch']}")
print(f"  {len(data_mc56['spike_times']):,} spikes")
print(f"  {data_mc56['n_trains']} trains, {data_mc56['train_amp'][0]:.0f} uA, {data_mc56['train_freq'][0]:.0f} Hz")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6),
                              sharex=True, height_ratios=[2, 1])
plot_raster_and_psth(data_mc56, window=(-0.5, 1.5), bin_size=0.02, sigma_ms=30,
    title='SE1 -> MC56 (C1): High-activity channel',
    ax_raster=ax1, ax_psth=ax2)
plt.show()


In [ ]:
plot_pulse_locked(data_mc56, window_ms=(-2, 9), bin_size_ms=0.25,
    max_pulses=800,
    title='SE1 -> MC56: Pulse-locked raster (monosynaptic detection)')
plt.show()


---
## Example 2: Biphasic response — SE1 x MC27

MC27 shows a dramatic **excitatory onset burst** within the first ~200 ms of
stimulation, followed by **sustained suppression** during the remainder of the
1-s train.  This biphasic pattern — excitation then inhibition — is commonly
observed and likely reflects an initial volley via the direct S1->M1 pathway
followed by inhibitory network effects.


In [ ]:
data_mc27 = load_channel_pair(DATA_DIR, stim_elec=1, motor_ch=27)
print(f"SE1 x MC27: {len(data_mc27['spike_times']):,} spikes")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6),
                              sharex=True, height_ratios=[2, 1])
plot_raster_and_psth(data_mc27, window=(-0.5, 1.5), bin_size=0.02, sigma_ms=30,
    title='SE1 -> MC27 (C1): Biphasic — excitatory onset, then suppression',
    ax_raster=ax1, ax_psth=ax2)
plt.show()


In [ ]:
plot_pulse_locked(data_mc27, window_ms=(-2, 9), bin_size_ms=0.25,
    max_pulses=600,
    title='SE1 -> MC27: Pulse-locked raster')
plt.show()


---
## Example 3: Inhibitory response — SE1 x MC129

MC129 shows the **opposite** pattern: firing is **suppressed** during the 1-s
ICMS train, followed by a clear **rebound excitation** after stim offset.
The paper found that ~48% of modulated M1 channels exhibit both excitation
and inhibition depending on which S1 electrode is stimulated.


In [ ]:
data_mc129 = load_channel_pair(DATA_DIR, stim_elec=1, motor_ch=129)
print(f"SE1 x MC129: {len(data_mc129['spike_times']):,} spikes")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6),
                              sharex=True, height_ratios=[2, 1])
plot_raster_and_psth(data_mc129, window=(-0.5, 1.5), bin_size=0.02, sigma_ms=40,
    title='SE1 -> MC129 (C1): Inhibitory — suppression during stim, rebound after',
    ax_raster=ax1, ax_psth=ax2)
plt.show()


---
## Example 4: Same M1 neuron, different S1 electrodes

A key finding is that the spatial pattern of M1 activation varies
systematically with which S1 electrode is stimulated, following somatotopic
maps.  Below we compare MC56's response to **SE1** vs **SE10**.


In [ ]:
data_se10_mc56 = load_channel_pair(DATA_DIR, stim_elec=10, motor_ch=56)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

plot_raster_and_psth(data_mc56, window=(-0.5, 1.5), bin_size=0.02, sigma_ms=30,
    title='SE1 -> MC56', ax_raster=axes[0, 0], ax_psth=axes[1, 0])
plot_raster_and_psth(data_se10_mc56, window=(-0.5, 1.5), bin_size=0.02, sigma_ms=30,
    title='SE10 -> MC56', ax_raster=axes[0, 1], ax_psth=axes[1, 1])

fig.suptitle('Same M1 channel, different S1 stimulation electrodes',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---
## Multi-channel overview: SE1 across the M1 array

This panel plots the SDF for 10 M1 channels during SE1 stimulation, ordered
by response type.  It illustrates the **heterogeneity of S1->M1 effects**:
some channels are strongly excited, some suppressed, some biphasic, and some
barely affected.


In [ ]:
channels = [56, 46, 43, 37, 27, 129, 39, 13, 98, 107]
labels   = ['MC56 (tonic)', 'MC46', 'MC43', 'MC37',
            'MC27 (biphasic)', 'MC129 (inhib)', 'MC39', 'MC13', 'MC98', 'MC107']

window   = (-0.5, 1.5)
bin_size = 0.02
sigma_bins = (30 / 1000) / bin_size

fig, axes = plt.subplots(len(channels), 1, figsize=(10, 1.8 * len(channels)),
                         sharex=True)

for idx, (mc, lab) in enumerate(zip(channels, labels)):
    try:
        d  = load_channel_pair(DATA_DIR, stim_elec=1, motor_ch=mc)
        sp = d['spike_times']
        ts = d['train_starts']
        nt = d['n_trains']

        aligned = np.concatenate([
            sp[(sp >= t0 + window[0]) & (sp < t0 + window[1])] - t0
            for t0 in ts])

        bins       = np.arange(window[0], window[1] + bin_size, bin_size)
        counts, be = np.histogram(aligned, bins=bins)
        rate       = counts / (bin_size * nt)
        bc         = (be[:-1] + be[1:]) / 2
        sdf        = gaussian_filter1d(rate.astype(float), sigma=sigma_bins)

        axes[idx].fill_between(bc, sdf, alpha=0.25, color='steelblue')
        axes[idx].plot(bc, sdf, 'b-', lw=1.2)
        axes[idx].axvline(0, color='red', ls='--', alpha=0.5)
        axes[idx].axvspan(0, 1.0, color='red', alpha=0.04)
        axes[idx].set_ylabel(f'{lab}\n({len(sp):,} sp)', fontsize=8)
        axes[idx].set_xlim(window)
    except Exception as e:
        axes[idx].text(0.5, 0.5, str(e), transform=axes[idx].transAxes, fontsize=8)

axes[-1].set_xlabel('Time relative to stim onset (s)')
axes[0].set_title('SE1 stimulation: SDF across M1 channels (Participant C1)',
                  fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Summary

| Observation | Example channel |
|-------------|-----------------|
| **Tonic excitation** during stim | MC56 — sustained rate increase |
| **Biphasic** (onset burst then suppression) | MC27 — excitatory transient, then inhibition |
| **Inhibition + rebound** | MC129 — suppressed during stim, rebound after offset |
| **Somatotopic specificity** | MC56 responds differently to SE1 vs SE10 |
| **Pulse-locked responses** (2-7 ms) | Possible monosynaptic S1->M1 activation |

### Key takeaways from the paper
- S1 ICMS modulates most M1 channels, with both excitatory and inhibitory effects
- A subset of M1 neurons show short-latency, pulse-locked responses (2-7 ms), consistent with monosynaptic connections
- The spatial pattern of M1 activation follows **somatotopic maps** — S1 electrodes with projected fields on a given finger preferentially activate M1 neurons involved in that finger's movement
- The effects are **task-dependent**: the same ICMS can produce different M1 responses depending on ongoing motor behaviour

### Reference
Shelchkova, N.D., Downey, J.E., Greenspon, C.M. et al. *Microstimulation of human somatosensory cortex evokes task-dependent, spatially patterned responses in motor cortex.* Nat Commun **14**, 7368 (2023). [doi:10.1038/s41467-023-43140-2](https://doi.org/10.1038/s41467-023-43140-2)
